# Bias checking

Load the normalized 12-feature start-state observation from every sample run.

In [1]:
from pathlib import Path

%cd ../..

d:\User Installed Folders\Master Thesis


In [2]:
from pathlib import Path
import math

import numpy as np

from Simulation.Auxilaries.io import load_runs
from Simulation.Auxilaries.sim_config import sim_config


def load_start_states(samples_path=Path("Explanations/Samples")) -> np.ndarray:
    """Return one normalized 12-feature start state per run."""
    states = []
    map_min = np.asarray(sim_config.boundary[0], dtype=float)
    map_size = np.asarray(sim_config.boundary[1], dtype=float) - map_min

    for run in load_runs(samples_path, sim_config):
        env, ship = run.scenario
        end_pos_norm = (
            (np.asarray(env.end_poly.centroid.coords[0]) - map_min) / map_size
        ) * 2 - 1
        ship_pos_norm = ((ship.state.position - map_min) / map_size) * 2 - 1
        rel_pos_norm = (end_pos_norm - ship_pos_norm) / 2
        rel_angle = (
            math.atan2(rel_pos_norm[1], rel_pos_norm[0]) - ship.state.heading
        )

        states.append([
            ship_pos_norm[0],
            ship_pos_norm[1],
            math.cos(ship.state.heading),
            math.sin(ship.state.heading),
            ship.state.velocity / ship.params.max_velocity,
            0.0,  # Every scenario starts at simulation step zero.
            rel_pos_norm[0],
            rel_pos_norm[1],
            math.cos(rel_angle),
            math.sin(rel_angle),
            end_pos_norm[0],
            end_pos_norm[1],
        ])

    return np.asarray(states, dtype=np.float32).reshape(-1, 12)

start_states = np.load("Explanations/Start/start_samples_10000.npy")
start_states.shape

(10000, 12)

In [4]:
import pandas as pd

from Agent.Observation.observation import feature_names


def summarize_feature_biases(states: np.ndarray) -> pd.DataFrame:
    """Summarize size and directional balance for every input feature."""
    states = np.asarray(states, dtype=float)
    if states.ndim != 2 or states.shape[1] != 12:
        raise ValueError(f"Expected states with shape (n, 12), got {states.shape}.")

    rows = []
    for column, name in enumerate(feature_names(pretty=True)):
        values = states[:, column]
        center = np.mean(values)
        median = np.median(values)
        std = np.std(values, ddof=1) if len(values) > 1 else 0.0
        centered = values - center
        skewness = (
            np.mean(centered ** 3) / np.std(values) ** 3
            if np.std(values) > 0 else np.nan
        )
        nonzero = values != 0
        positive_fraction = np.mean(values > 0)
        negative_fraction = np.mean(values < 0)
        nonzero_count = np.count_nonzero(nonzero)

        rows.append({
            "feature": name,
            "mean": center,
            "median": median,
            "mean_abs": np.mean(np.abs(values)),
            "std": std,
            "mad": np.median(np.abs(values - median)),
            "q05": np.quantile(values, 0.05),
            "q25": np.quantile(values, 0.25),
            "q75": np.quantile(values, 0.75),
            "q95": np.quantile(values, 0.95),
            "skewness": skewness,
            "positive_fraction": positive_fraction,
            "negative_fraction": negative_fraction,
            "zero_fraction": np.mean(~nonzero),
            "sign_imbalance": (
                abs(np.count_nonzero(values > 0) - np.count_nonzero(values < 0))
                / nonzero_count
                if nonzero_count else 0.0
            ),
            "mean_ci95_low": center - 1.96 * std / np.sqrt(len(values)),
            "mean_ci95_high": center + 1.96 * std / np.sqrt(len(values)),
        })

    return pd.DataFrame(rows).set_index("feature")


bias_metrics = summarize_feature_biases(start_states)
bias_metrics

,mean,median,mean_abs,std,mad,q05,q25,q75,q95,skewness,positive_fraction,negative_fraction,zero_fraction,sign_imbalance,mean_ci95_low,mean_ci95_high
feature,,,,,,,,,,,,,,,,
Ship X Norm,-2.580537e-02,-0.071734,6.783580e-01,7.315660e-01,0.820266,-8.920000e-01,-0.860448,0.824800,8.920000e-01,0.060287,0.4813,0.5187,0.0000,0.037400,-4.014407e-02,-1.146668e-02
Ship Y Norm,-1.667461e-02,-0.055955,6.747951e-01,7.273474e-01,0.824329,-8.920000e-01,-0.842112,0.803465,8.920000e-01,0.037753,0.4865,0.5135,0.0000,0.027000,-3.093062e-02,-2.418605e-03
Heading Cos,2.202318e-02,0.023842,6.475726e-01,7.098197e-01,0.714038,-9.825618e-01,-0.696696,0.732132,9.871335e-01,-0.043782,0.5164,0.4836,0.0000,0.032800,8.110709e-03,3.593564e-02
Heading Sin,1.585034e-02,0.025602,6.406658e-01,7.039315e-01,0.697957,-9.845977e-01,-0.685923,0.708481,9.832628e-01,-0.044889,0.5173,0.4820,0.0007,0.035325,2.053281e-03,2.964740e-02
Velocity Norm,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,NaN,0.0000,0.0000,1.0000,0.000000,0.000000e+00,0.000000e+00
Time Elapsed Norm,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,NaN,0.0000,0.0000,1.0000,0.000000,0.000000e+00,0.000000e+00
Rel X Norm,2.185945e-02,0.020522,5.984865e-01,6.598202e-01,0.673020,-8.920000e-01,-0.649833,0.694637,8.920000e-01,-0.045998,0.5085,0.4818,0.0097,0.026962,8.926973e-03,3.479193e-02
Rel Y Norm,1.560891e-02,0.021635,5.914665e-01,6.531857e-01,0.662021,-8.920000e-01,-0.652204,0.669770,8.920000e-01,-0.038940,0.5143,0.4794,0.0063,0.035121,2.806474e-03,2.841135e-02
Rel Angle Cos,1.000000e+00,1.000000,1.000000e+00,0.000000e+00,0.000000,1.000000e+00,1.000000,1.000000,1.000000e+00,NaN,1.0000,0.0000,0.0000,1.000000,1.000000e+00,1.000000e+00
